In [ ]:
!pip install tf2onnx

In [ ]:
!pip install esp-ppq

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model('sports.keras')
model.summary()

In [ ]:
import tf2onnx

spec = (tf.TensorSpec((None, 224, 224, 3), tf.float32, name="input_image"),)
model.output_names = ['final_output']
tf2onnx.convert.from_keras(model, input_signature=spec, opset=13, output_path="sports.onnx")

In [ ]:
import kagglehub

path = kagglehub.dataset_download('gpiosenka/sports-classification')

In [ ]:
import os

train_data = tf.keras.utils.image_dataset_from_directory(
    os.path.join(path, 'train'),
    labels="inferred",
    label_mode="int",
    color_mode="rgb",
    batch_size=32,
    image_size=(224, 224),
    shuffle=True,
)

In [ ]:
from torch.utils.data import DataLoader
from esp_ppq.api import espdl_quantize_onnx
from torchvision import datasets, transforms
import torch

calib_batches = []
print("Converting TF batches to Torch...")

# We only need as many batches as calib_steps (e.g., 32)
for images, labels in train_data.take(32):
    # Convert to float32 and move to torch
    torch_batch = torch.from_numpy(images.numpy().astype('float32'))
    calib_batches.append(torch_batch)

# 3. Dummy collate because the data is already batched
def collate_fn(batch):
    return batch


ONNX_MODEL_PATH = "sports.onnx"
ESPDL_MODEL_PATH = "sports.espdl"
INPUT_SHAPE = [32, 224, 224, 3]
TARGET = "esp32s3"
NUM_OF_BITS = 16
DEVICE = "cpu"

quant_ppq_graph = espdl_quantize_onnx(
    onnx_import_file=ONNX_MODEL_PATH,
    espdl_export_file=ESPDL_MODEL_PATH,
    calib_dataloader=calib_batches,
    calib_steps=len(calib_batches),
    input_shape=INPUT_SHAPE,
    inputs=None,
    target=TARGET,
    num_of_bits=NUM_OF_BITS,
    collate_fn=collate_fn,
    dispatching_override=None,
    device=DEVICE,
    error_report=True,
    skip_export=False,
    export_test_values=True,
    verbose=1,
)